# CareerCraft - live pipeline demo

This notebook runs the **real** CareerCraft pipeline end to end, importing the same
modules the MCP servers import. Nothing here is re-implemented for the demo:

| Stage | Module actually called |
| --- | --- |
| Resume parsing | `server.app.services.resume_parser.parse_resume_file` |
| Job matching | `server.app.services.matching_engine.rank_jobs_for_resume` |
| Cover letter | `server.app.services.cover_letter_generator.generate_cover_letter` |

**What is real and what is fixture**

- The resume parser, the sentence-transformers matching engine and the LLM call are
  all executed for real. The cover letter below was written by a local Ollama model -
  the cell prints the model name and states whether the call was live or fell back to
  a clearly-labelled placeholder.
- The candidate (*Alex Rivera*, `sample_resume.txt`) is **synthetic**, so no real
  personal data is committed to the repo.
- The job postings are a small **fixed sample set** defined in this notebook rather
  than a live Jobicy fetch, so the notebook is deterministic and runs offline. The
  live fetch is one call away and is shown (unexecuted) at the end.

No cloud API key is needed anywhere - the LLM is a local Ollama daemon.

## 0. Environment

In [1]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

# Run from anywhere: put the repo root on sys.path so `server.app...` resolves.
ROOT = Path.cwd()
if not (ROOT / "server").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

import numpy, spacy, ollama
print("python  :", sys.version.split()[0])
print("numpy   :", numpy.__version__)
print("spaCy   :", spacy.__version__)

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434")
try:
    installed = [m.model for m in ollama.Client(host=OLLAMA_BASE_URL).list().models]
    print("ollama  :", OLLAMA_BASE_URL, "->", installed or "no models installed")
except Exception as exc:
    installed = []
    print("ollama  :", OLLAMA_BASE_URL, "-> unreachable:", type(exc).__name__, exc)

repo root: C:\Users\thoma\Downloads\ai-final-project\job-mcp-agent


python  : 3.13.7
numpy   : 2.3.3
spaCy   : 3.8.11


ollama  : http://127.0.0.1:11434 -> ['llama3.1:8b', 'qwen2.5:1.5b', 'tinyllama:latest', 'llama3.2:1b', 'llama3.2:latest']


## 1. Parse the resume

`parse_resume_file` accepts PDF, DOCX or TXT. For PDFs it segments sections from the
PyMuPDF layout (font size and weight) and falls back to OCR when a page yields almost
no text; here we hand it the plain-text sample so the repo carries no binary fixture.

Skills come from a spaCy `PhraseMatcher` over a curated vocabulary, applied
sentence-by-sentence to cut false positives.

In [2]:
import json
from server.app.services.resume_parser import parse_resume_file

RESUME_PATH = ROOT / "notebooks" / "sample_resume.txt"
resume = parse_resume_file(RESUME_PATH)

print("name        :", resume["name"])
print("contacts    :", json.dumps(resume["contacts"]))
print("skills (%d) :" % resume["skills_count"], ", ".join(resume["skills"]))
print("sections    :", ", ".join(resume["sections_detected"]))
print("warnings    :", resume["parse_warnings"] or "none")
print()
for role in resume["experience"]:
    print(f'  {role["title"]} - {role["organization"]}  ({role["period"]})')
    for h in role["highlights"]:
        print("     *", h[:94] + ("..." if len(h) > 94 else ""))
print()
for edu in resume["education"]:
    print("  education:", edu["text"])
    print("             degree:", edu["degree"], "| years:", edu["years"])
print()
for proj in resume["projects"]:
    print("  project:", proj["name"])

name        : Alex Rivera
contacts    : {"email": "alex.rivera@example.com", "phone": "+1 415 555 0142", "linkedin": "https://linkedin.com/in/alex-rivera-demo", "github": "https://github.com/alex-rivera-demo"}
skills (18) : airflow, aws, docker, fastapi, git, kubernetes, linux, numpy, pandas, postgresql, python, pytorch, regex, scikit-learn, snowflake, spacy, spark, sql
sections    : profile, skills, experience, education, academic projects
warnings    : none

  Data Engineer - Northwind Analytics, San Francisco, CA  (Jun 2023 - Present)
     * Rebuilt the nightly ingestion pipeline on Airflow and Spark and cut the batch window from six ...
     * Designed the dimensional warehouse model in Snowflake that now backs every executive reporting...
     * Added contract tests and column-level lineage checks that catch schema drift long before it ev...
  Machine Learning Intern - Cascade Robotics, Seattle, WA  (Jun 2022 - Sep 2022)
     * Trained a PyTorch defect-classification model on prod

## 2. The job postings

Field names follow the Jobicy schema (`jobTitle`, `companyName`, `jobGeo`, `jobLevel`,
`jobDescription`) because that is what `get_data.JobicyProvider` returns; the matching
engine also accepts the normalised `title` / `company` / `location` aliases.

Two of the eight are deliberate negatives - a senior architect role and a PhD-gated
research role - so we can watch the pre-filters reject them.

In [3]:
SAMPLE_JOBS = [
    {"id": 1, "jobTitle": "Data Engineer, Analytics Platform", "companyName": "Meridian Health",
     "jobGeo": "Remote - USA", "jobLevel": "Mid", "jobType": "Full-time",
     "jobDescription": "Own batch and streaming pipelines built on Airflow, Spark and "
                       "Snowflake. You will model warehouse tables, add data quality "
                       "contracts, and partner with analysts who depend on the nightly "
                       "load. Python and SQL every day."},
    {"id": 2, "jobTitle": "Machine Learning Engineer", "companyName": "Foundry Vision",
     "jobGeo": "Remote - USA", "jobLevel": "Mid", "jobType": "Full-time",
     "jobDescription": "Train and ship computer vision models in PyTorch for industrial "
                       "inspection. Package models behind FastAPI services, containerize "
                       "with Docker, and deploy to edge hardware in customer factories."},
    {"id": 3, "jobTitle": "Backend Engineer, Platform APIs", "companyName": "Latch Systems",
     "jobGeo": "Remote - USA", "jobLevel": "Mid", "jobType": "Full-time",
     "jobDescription": "Build Python services with FastAPI and PostgreSQL. Kubernetes on "
                       "AWS, strong testing culture, and a team that reviews every "
                       "migration carefully."},
    {"id": 4, "jobTitle": "Senior Staff Data Architect", "companyName": "Corvus Capital",
     "jobGeo": "New York, NY", "jobLevel": "Senior", "jobType": "Full-time",
     "jobDescription": "Lead the data architecture practice across the firm. Twelve or "
                       "more years of experience designing warehouses at scale, and a "
                       "track record mentoring principal engineers."},
    {"id": 5, "jobTitle": "Research Scientist, Representation Learning",
     "companyName": "Nord Institute",
     "jobGeo": "Boston, MA", "jobLevel": "Senior", "jobType": "Full-time",
     "jobDescription": "PhD in machine learning or a closely related field required. "
                       "Publish at top venues and advance our doctorate-level research "
                       "agenda in self-supervised representation learning."},
    {"id": 6, "jobTitle": "Analytics Engineer", "companyName": "Bramble Retail",
     "jobGeo": "Remote - USA", "jobLevel": "Junior", "jobType": "Full-time",
     "jobDescription": "Write dbt models over a Snowflake warehouse, maintain SQL tests, "
                       "and help the analytics team keep the semantic layer trustworthy. "
                       "Some Python scripting."},
    {"id": 7, "jobTitle": "Technical Writer, Developer Docs", "companyName": "Pelagic Cloud",
     "jobGeo": "Remote - USA", "jobLevel": "Mid", "jobType": "Full-time",
     "jobDescription": "Write reference and tutorial documentation for our developer "
                       "platform. Interview engineers, edit for clarity, and own the docs "
                       "style guide."},
    {"id": 8, "jobTitle": "Registered Nurse, Paediatrics", "companyName": "Rowan Medical",
     "jobGeo": "Denver, CO", "jobLevel": "Mid", "jobType": "Full-time",
     "jobDescription": "Provide direct patient care on the paediatric ward. Active RN "
                       "licence and two years of clinical experience required."},
]

import pandas as pd
pd.DataFrame(SAMPLE_JOBS)[["id", "jobTitle", "companyName", "jobLevel", "jobGeo"]]

,id,jobTitle,companyName,jobLevel,jobGeo
0,1,"Data Engineer, Analytics Platform",Meridian Health,Mid,Remote - USA
1,2,Machine Learning Engineer,Foundry Vision,Mid,Remote - USA
2,3,"Backend Engineer, Platform APIs",Latch Systems,Mid,Remote - USA
3,4,Senior Staff Data Architect,Corvus Capital,Senior,"New York, NY"
4,5,"Research Scientist, Representation Learning",Nord Institute,Senior,"Boston, MA"
5,6,Analytics Engineer,Bramble Retail,Junior,Remote - USA
6,7,"Technical Writer, Developer Docs",Pelagic Cloud,Mid,Remote - USA
7,8,"Registered Nurse, Paediatrics",Rowan Medical,Mid,"Denver, CO"


## 3. Rank the jobs

`rank_jobs_for_resume` runs in two phases.

1. **Heuristic pre-filter.** It infers whether the candidate reads as junior and what
   their highest degree is, then drops obviously senior postings and postings demanding
   a degree above what the resume shows.
2. **Semantic scoring.** The surviving postings and the resume are encoded with
   `all-MiniLM-L6-v2` and ranked by cosine similarity, keeping anything at or above
   `min_similarity`.

The first call loads the MiniLM weights, so it takes a few seconds.

In [4]:
from server.app.services.matching_engine import (
    rank_jobs_for_resume,
    _candidate_looks_junior,
    _highest_degree_level,
    _job_is_obviously_senior,
    _job_requires_masters,
    _job_requires_phd,
)

print("candidate reads as junior :", _candidate_looks_junior(resume))
print("highest degree level (0-3):", _highest_degree_level(resume))

matches = rank_jobs_for_resume(
    resume,
    SAMPLE_JOBS,
    top_k=10,
    min_similarity=0.25,
    filter_senior_for_grads=True,
)

pd.DataFrame(
    [
        {
            "rank": i,
            "similarity": round(j["similarity"], 3),
            "title": j["jobTitle"],
            "company": j["companyName"],
            "level": j["jobLevel"],
        }
        for i, j in enumerate(matches, 1)
    ]
).set_index("rank")

candidate reads as junior : True
highest degree level (0-3): 0


,similarity,title,company,level
rank,,,,
1,0.622,"Data Engineer, Analytics Platform",Meridian Health,Mid
2,0.608,Machine Learning Engineer,Foundry Vision,Mid
3,0.534,Analytics Engineer,Bramble Retail,Junior
4,0.447,"Backend Engineer, Platform APIs",Latch Systems,Mid
5,0.330,"Technical Writer, Developer Docs",Pelagic Cloud,Mid


### Why the rejected postings were rejected

In [5]:
kept_ids = {j["id"] for j in matches}
for job in SAMPLE_JOBS:
    if job["id"] in kept_ids:
        continue
    reasons = []
    if _job_is_obviously_senior(job):
        reasons.append("pre-filter: reads as a senior role")
    if _job_requires_phd(job):
        reasons.append("pre-filter: requires a PhD")
    if _job_requires_masters(job):
        reasons.append("pre-filter: requires a Master's")
    if not reasons:
        reasons.append("scored below the 0.25 similarity floor")
    print(f'{job["id"]}. {job["jobTitle"]} @ {job["companyName"]}')
    for reason in reasons:
        print("     -", reason)

4. Senior Staff Data Architect @ Corvus Capital
     - pre-filter: reads as a senior role
5. Research Scientist, Representation Learning @ Nord Institute
     - pre-filter: reads as a senior role
     - pre-filter: requires a PhD
8. Registered Nurse, Paediatrics @ Rowan Medical
     - scored below the 0.25 similarity floor


## 4. Generate a cover letter

`generate_cover_letter` condenses the parsed resume, formats the posting, and sends a
system + human message pair to a local Ollama model through LangChain's `ChatOllama`.
`tone` (professional / enthusiastic / concise / academic) and `length`
(short / medium / long) steer the prompt.

The service raises rather than silently returning boilerplate when Ollama is
unreachable. The cell below catches that and substitutes an obviously-labelled
placeholder so the notebook still renders - read the banner it prints to see which of
the two actually happened.

In [6]:
import textwrap
import time

from server.app.services.cover_letter_generator import generate_cover_letter

MODEL_NAME = "llama3.2:1b"
target_job = matches[0]

STUBBED = False
failure = None
started = time.time()
try:
    letter = generate_cover_letter(
        resume,
        target_job,
        model_name=MODEL_NAME,
        temperature=0.7,
        tone="professional",
        length="medium",
    )
except Exception as exc:
    STUBBED = True
    failure = f"{type(exc).__name__}: {exc}"
    letter = (
        "Dear Hiring Manager,\n\n"
        "[Placeholder written by the notebook - NOT model output. The live Ollama call "
        "failed, so no cover letter was generated. Start the daemon with `ollama serve`, "
        f"pull the model with `ollama pull {MODEL_NAME}`, then re-run this cell to see a "
        "real letter here.]\n\n"
        "Sincerely,\nAlex Rivera"
    )
elapsed = time.time() - started

if STUBBED:
    print("=" * 78)
    print("!!  STUBBED PLACEHOLDER - NOT A REAL MODEL RESPONSE  !!")
    print("The text below was written by this notebook, not by an LLM.")
    print("Reason:", failure)
    print("=" * 78)
else:
    print("=" * 78)
    print(f"LIVE model output - generated by Ollama model '{MODEL_NAME}'")
    print(f"endpoint {OLLAMA_BASE_URL}  |  {elapsed:.1f}s  |  {len(letter)} characters")
    print(f"for: {target_job['jobTitle']} @ {target_job['companyName']}")
    print("=" * 78)

print()
for para in letter.split("\n"):
    print(textwrap.fill(para, width=78) if para.strip() else "")

C:\Users\thoma\Downloads\ai-final-project\job-mcp-agent\server\app\services\cover_letter_generator.py:266: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(


LIVE model output - generated by Ollama model 'llama3.2:1b'
endpoint http://127.0.0.1:11434  |  105.2s  |  2284 characters
for: Data Engineer, Analytics Platform @ Meridian Health

Here's a tailored cover letter that addresses the candidate's background and
highlights their relevance for the Data Engineer, Analytics Platform role at
Meridian Health:

As a seasoned data engineer with a passion for building scalable, high-
performance pipelines, I am thrilled to apply for the Data Engineer position
at Meridian Health. With my expertise in designing and implementing complex
batch and streaming architectures on Airflow, Spark, and Snowflake, I am
confident that I can drive significant performance gains in your analytics
platform.

My experience as a data engineer at Northwind Analytics has provided me with a
deep understanding of the importance of dimensional warehouse models in
supporting executive-level reporting. In my current role, I have successfully
rebuilt the nightly ingestion pipe

## 5. The same pipeline across the MCP boundary

Everything above called the services directly, which is what keeps this notebook
self-contained. In the running system the web frontend never does that - it goes through
an MCP tool call. Here is the real tool catalog, read straight off the FastMCP server
object rather than transcribed by hand.

In [7]:
from server.mcp_pipeline_server import mcp

# The notebook kernel already runs an event loop, so await at top level.
tools = await mcp.get_tools()

for tool_name, tool in tools.items():
    schema = tool.parameters or {}
    required = set(schema.get("required", []))
    params = ", ".join(
        p + ("" if p in required else "=...") for p in schema.get("properties", {})
    )
    summary = (tool.description or "").strip().splitlines()
    print(f"{tool_name}({params})")
    print("   ", summary[0] if summary else "")

C:\Users\thoma\AppData\Local\Programs\Python\Python313\Lib\site-packages\fastmcp\server\auth\providers\jwt.py:10: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import JsonWebKey, JsonWebToken


fetch_job_data(count=..., out_path=...)
    Fetch job listings from the API and save to a JSON file.
populate_mongodb(out_path=..., mongo_url=...)
    Populate MongoDB with jobs from a JSON file.
parse_resume(resume_path)
    Parse a resume file (PDF, DOCX, or TXT) into structured data.
create_cover_letter(resume, job, model_name=..., temperature=..., tone=..., length=...)
    Generate a personalized cover letter for a job posting.
match_jobs_to_resume(resume, jobs_source=..., jobs_file=..., mongo_url=..., top_k=..., min_similarity=..., filter_senior_for_grads=..., model_name=...)
    Match and rank jobs for a parsed resume using semantic similarity.
match_jobs_from_resume_path(resume_path, jobs_source=..., jobs_file=..., mongo_url=..., top_k=..., min_similarity=..., filter_senior_for_grads=..., model_name=...)
    Parse resume and match jobs in one step.
run_complete_pipeline(resume_path, job_count=..., jobs_file=..., mongo_url=..., generate_cover_letter_for_first=...)
    Execute the

To drive those tools over HTTP instead, start the server and use a `fastmcp` client.
This is left unexecuted because it needs a second process listening on `:8002`:

```python
# terminal:  python server/mcp_pipeline_server.py
from fastmcp import Client

async with Client("http://127.0.0.1:8002/mcp") as client:
    result = await client.call_tool("run_complete_pipeline", {
        "resume_path": "notebooks/sample_resume.txt",
        "job_count": 50,
        "generate_cover_letter_for_first": True,
    })
```

And to rank against live postings instead of the fixed sample set, swap step 2 for:

```python
from get_data import fetch_jobs
jobs = fetch_jobs(count=50)          # hits the Jobicy API, writes jobs.json
```

## Recap

The parse, the ranking and the cover letter above all came from the production modules.
The pre-filters removed the senior and PhD-gated postings before any embedding was
computed, MiniLM ordered what was left by cosine similarity against the resume, and a
local Ollama model wrote the letter for the top match - with no cloud API key involved
at any point.